# Your First GAM with Aurora-GLM

## Overview

This quickstart notebook demonstrates how to fit your first Generalized Additive Model (GAM) using Aurora-GLM. We'll model temperature over time with smooth seasonal patterns.

## What You'll Learn

- Generate data with non-linear relationships
- Fit a GAM with smooth terms
- Understand GCV (Generalized Cross-Validation)
- Visualize smooth curves
- Compare GAM vs GLM

---

## 1. Setup

Import necessary libraries:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aurora.models import fit_glm
from aurora.models.gam import fit_gam

# Set random seed for reproducibility
np.random.seed(42)

## 2. Generate Synthetic Data

Create temperature data with seasonal pattern:
- Day of year (1-365)
- Temperature follows sinusoidal pattern (seasonal)
- Added noise

In [ ]:
# Generate 365 days
n = 365
days = np.arange(1, n + 1)

# True seasonal pattern: sinusoidal with peak in summer (day 180)
# Temperature = 15 (baseline) + 10 * sin(2π * (day - 90) / 365)
true_temp = 15 + 10 * np.sin(2 * np.pi * (days - 90) / 365)

# Add noise
temperature = true_temp + np.random.randn(n) * 2

# Create DataFrame
df = pd.DataFrame({
    'day': days,
    'temperature': temperature,
    'true_temp': true_temp
})

print(f"Generated {len(df)} days of temperature data")

# Plot raw data
plt.figure(figsize=(10, 4))
plt.scatter(df['day'], df['temperature'], alpha=0.3, s=10, label='Observed')
plt.plot(df['day'], df['true_temp'], 'r-', lw=2, label='True pattern')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title('Annual Temperature Pattern')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Prepare Data for Modeling

Create design matrix and response vector:

In [ ]:
# Response vector
y = df['temperature'].values

# Predictor (will be smoothed)
X_raw = df['day'].values

print(f"y shape: {y.shape}")
print(f"X_raw shape: {X_raw.shape}")

## 4. Comparison: GLM vs GAM

First, let's try a simple linear model (GLM) to see why we need GAM:

In [ ]:
# Fit linear model (GLM)
X_linear = np.column_stack([np.ones(n), X_raw])
result_glm = fit_glm(X=X_linear, y=y, family='gaussian')

# Predictions
y_pred_linear = result_glm.predict(X_linear)

# Plot
plt.figure(figsize=(10, 4))
plt.scatter(df['day'], df['temperature'], alpha=0.3, s=10, label='Observed')
plt.plot(df['day'], y_pred_linear, 'g--', lw=2, label='Linear fit (GLM)')
plt.plot(df['day'], df['true_temp'], 'r-', lw=2, label='True pattern')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title('GLM: Linear Model Cannot Capture Seasonal Pattern')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"GLM R² = {1 - np.sum((y - y_pred_linear)**2) / np.sum((y - y.mean())**2):.4f}")

**Problem:** Linear model cannot capture the seasonal (sinusoidal) pattern!

## 5. Fit GAM with Smooth Term

Now fit a GAM with a smooth function of day:

In [ ]:
# Fit GAM with smooth term for 'day'
# fit_gam automatically creates the basis and selects smoothing parameter
result_gam = fit_gam(
    x=X_raw,
    y=y,
    n_basis=20,
    basis_type='bspline',
    degree=3
)

print(f"GAM Results:")
print(f"Lambda (smoothing parameter): {result_gam.lambda_:.2e}")
print(f"EDF (effective degrees of freedom): {result_gam.edf:.2f}")
print(f"GCV score: {result_gam.gcv_score:.4f}")

**Key Concepts:**

- **Smooth term**: Flexible curve instead of straight line
- **Lambda (λ)**: Smoothing parameter (higher = smoother curve)
- **GCV**: Automatically selects optimal λ to balance fit and smoothness
- **EDF**: Effective degrees of freedom (complexity of the smooth)

## 6. Visualize GAM Fit

Plot the smooth curve:

In [ ]:
# Get fitted values
y_pred_gam = result_gam.fitted_values

# Plot comparison
plt.figure(figsize=(12, 5))

# Subplot 1: GAM fit
plt.subplot(1, 2, 1)
plt.scatter(df['day'], df['temperature'], alpha=0.3, s=10, label='Observed')
plt.plot(df['day'], y_pred_gam, 'b-', lw=2, label='GAM fit')
plt.plot(df['day'], df['true_temp'], 'r--', lw=2, alpha=0.7, label='True pattern')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title('GAM: Captures Seasonal Pattern')
plt.legend()
plt.grid(alpha=0.3)

# Subplot 2: Comparison
plt.subplot(1, 2, 2)
plt.scatter(df['day'], df['temperature'], alpha=0.3, s=10, label='Observed')
plt.plot(df['day'], y_pred_linear, 'g--', lw=2, label='GLM (linear)')
plt.plot(df['day'], y_pred_gam, 'b-', lw=2, label='GAM (smooth)')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title('GLM vs GAM')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate R²
r2_glm = 1 - np.sum((y - y_pred_linear)**2) / np.sum((y - y.mean())**2)
r2_gam = result_gam.r_squared

print(f"\nModel Comparison:")
print(f"GLM R² = {r2_glm:.4f}")
print(f"GAM R² = {r2_gam:.4f}")
print(f"Improvement: {(r2_gam - r2_glm) * 100:.1f}% increase in explained variance")

## 7. Inspect Smooth Term

Visualize the smooth function itself (centered):

In [ ]:
# Extract smooth component
# The fitted values already represent the smooth function
smooth_contribution_centered = result_gam.fitted_values - result_gam.fitted_values.mean()

# Plot smooth term
plt.figure(figsize=(10, 4))
plt.plot(df['day'], smooth_contribution_centered, 'b-', lw=2)
plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Day of Year')
plt.ylabel('Smooth Effect (°C)')
plt.title(f'Smooth Term: s(day) [EDF = {result_gam.edf:.1f}]')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The smooth term shows the seasonal temperature deviation from the mean.")
print(f"Peak temperature occurs around day {df['day'][np.argmax(smooth_contribution_centered)]}.")

## 8. Make Predictions

Predict temperature for specific days:

In [ ]:
# New days to predict
new_days = np.array([1, 90, 180, 270, 365])  # Winter, Spring, Summer, Fall, Winter

# Predict using GAMResult.predict()
predictions = result_gam.predict(new_days)

# Display results
pred_df = pd.DataFrame({
    'day': new_days,
    'season': ['Winter', 'Spring', 'Summer', 'Fall', 'Winter'],
    'predicted_temp': predictions
})

print("\nPredictions for specific days:")
print(pred_df.to_string(index=False))

## Key Takeaways

- **GAM** = GLM + Smooth terms (non-linear relationships)
- **Smooth term** replaces linear predictor with flexible curve
- **GCV** automatically balances fit quality vs smoothness
- **EDF** measures complexity (higher = more wiggly curve)

## Next Steps

- **Mixed models:** Try `00_quickstart/03_first_gamm.ipynb` to add random effects
- **Multiple smooths:** See `02_classification/03_gam_classification.ipynb` for GAM with multiple smooth terms
- **Advanced GAM:** See `05_advanced_topics/01_tensor_smooths.ipynb` for 2D smoothing

## Resources

- [Aurora-GLM Documentation](https://github.com/Matcraft94/Aurora-GLM)
- [GAM Theory](https://en.wikipedia.org/wiki/Generalized_additive_model)
- [Wood (2017) - Generalized Additive Models: An Introduction with R](https://www.routledge.com/Generalized-Additive-Models-An-Introduction-with-R-Second-Edition/Wood/p/book/9781498728331)